# AeroPulse — Bronze Maintenance Auto Loader

## Purpose

Incrementally ingest maintenance events generated by the
Maintenance Application into the Bronze layer.

## Source System

Maintenance Application

## Entity

Maintenance Events

## Source Format

JSON

## Ingestion Technology

Databricks Auto Loader

## Streaming Source

cloudFiles

## Trigger

AvailableNow

## Target

workspace.aeropulse_dev.bronze_maintenance

## Operational Controls

- Auto Loader checkpoint
- Pipeline audit
- Source metadata
- Rerun safety
- Incremental ingestion

In [0]:
ENVIRONMENT = "dev"

CATALOG = "workspace"
SCHEMA = f"aeropulse_{ENVIRONMENT}"

SOURCE_SYSTEM = "maintenance_app"
SOURCE_ENTITY = "maintenance"

FILE_FORMAT = "json"

SOURCE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing/"
    f"{SOURCE_SYSTEM}/{SOURCE_ENTITY}"
)

CHECKPOINT_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/"
    f"raw_landing/_checkpoints/"
    f"{SOURCE_SYSTEM}/{SOURCE_ENTITY}"
)

BRONZE_TABLE = (
    f"{CATALOG}.{SCHEMA}.bronze_{SOURCE_ENTITY}"
)

print(f"Source      : {SOURCE_PATH}")
print(f"Checkpoint  : {CHECKPOINT_PATH}")
print(f"Bronze      : {BRONZE_TABLE}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

source_df = (
    spark.read
    .json(f"{SOURCE_PATH}/*")
)

schema_df = (
    source_df
    .withColumn("_rescued_data", F.lit(None).cast(StringType()))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_source_entity", F.lit(SOURCE_ENTITY))
    .withColumn("_source_file_path", F.lit(None).cast(StringType()))
)

schema_df.printSchema()

In [0]:
(
    schema_df
    .limit(0)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(
    f"Created Bronze table: {BRONZE_TABLE}"
)

In [0]:
print(
    spark.table(BRONZE_TABLE).count()
)

In [0]:
maintenance_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        CHECKPOINT_PATH
    )
    .load(SOURCE_PATH)
)

In [0]:
maintenance_stream_df.printSchema()

In [0]:
from pyspark.sql import functions as F

maintenance_bronze_df = (
    maintenance_stream_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "_source_entity",
        F.lit(SOURCE_ENTITY)
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
)

In [0]:
query = (
    maintenance_bronze_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        BRONZE_TABLE
    )
)

In [0]:
query.awaitTermination()

In [0]:
print(
    spark.table(BRONZE_TABLE).count()
)

In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {BRONZE_TABLE}
        LIMIT 20
    """)
)

In [0]:
display(
    dbutils.fs.ls(
        CHECKPOINT_PATH
    )
)

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/source_generators/maintenance_generator.py

In [0]:
from datetime import datetime, timezone

delivery_timestamp_3 = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_%f")

delivery_path_3 = (
    f"{SOURCE_PATH}/"
    f"maintenance_{delivery_timestamp_3}"
)

print(delivery_path_3)

In [0]:
(
    maintenance_df_3.write
    .mode("error")
    .json(delivery_path_3)
)

In [0]:
display(
    dbutils.fs.ls(
        SOURCE_PATH
    )
)

In [0]:
maintenance_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        CHECKPOINT_PATH
    )
    .load(SOURCE_PATH)
)

In [0]:
maintenance_bronze_df = (
    maintenance_stream_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "_source_entity",
        F.lit(SOURCE_ENTITY)
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
)

In [0]:
query = (
    maintenance_bronze_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        BRONZE_TABLE
    )
)

In [0]:
query.awaitTermination()

In [0]:
print(
    spark.table(BRONZE_TABLE).count()
)